<a href="https://colab.research.google.com/github/guthasushmasree/ML-LabPrograms/blob/main/ml(skill_7).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# EXPERIMENT
# REGRESSION, CATEGORICAL ENCODING AND FEATURE SCALING
#
# UNIVERSAL CSV VERSION - GOOGLE COLAB
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt

from google.colab import files

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.impute import SimpleImputer

from sklearn.linear_model import (
    LinearRegression,
    SGDRegressor
)

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)


# ============================================================
# 1. UPLOAD CSV
# ============================================================

print("=" * 70)
print("UNIVERSAL REGRESSION EXPERIMENT")
print("=" * 70)

print("\nUpload your CSV dataset.")

uploaded = files.upload()

if len(uploaded) == 0:
    raise ValueError("No CSV file was uploaded.")

file_path = list(uploaded.keys())[0]

print("\nSelected file:")
print(file_path)


# ============================================================
# 2. LOAD DATASET
# ============================================================

try:

    df = pd.read_csv(file_path)

except Exception as e:

    raise ValueError(
        f"Could not read CSV file:\n{e}"
    )


print("\nDataset loaded successfully!")

print("\nDataset shape:")
print(df.shape)

print("\nFirst 5 rows:")
print(df.head())

print("\nColumn names:")
print(df.columns.tolist())


# ============================================================
# 3. BASIC CLEANING
# ============================================================

# Remove completely empty columns

empty_columns = []

for col in df.columns:

    if df[col].isna().all():

        empty_columns.append(col)


if empty_columns:

    print("\nRemoving completely empty columns:")

    for col in empty_columns:

        print(" -", col)

    df = df.drop(
        columns=empty_columns
    )


# Remove duplicate rows

duplicate_count = df.duplicated().sum()

if duplicate_count > 0:

    print(
        f"\nRemoving {duplicate_count} duplicate rows."
    )

    df = df.drop_duplicates()


# Reset index

df = df.reset_index(drop=True)


print("\nShape after cleaning:")
print(df.shape)


# ============================================================
# 4. FIND POSSIBLE REGRESSION TARGET
# ============================================================

# Common regression target names

possible_targets = [

    "target",
    "y",
    "output",
    "result",

    "cgpa",
    "gpa",

    "price",
    "sale_price",
    "selling_price",

    "salary",
    "income",

    "score",
    "marks",
    "rating",

    "age",

    "weight",
    "height",

    "temperature",

    "profit",
    "revenue",
    "sales",

    "house_price",
    "houseprice",

    "final_score",
    "final_marks",

    "prediction",
    "value"
]


target_col = None


# ------------------------------------------------------------
# Search common target names
# ------------------------------------------------------------

for col in df.columns:

    cleaned = (
        str(col)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    if cleaned in possible_targets:

        target_col = col

        break


# ============================================================
# 5. IF TARGET NOT FOUND, FIND NUMERIC CANDIDATES
# ============================================================

if target_col is None:

    print(
        "\nNo standard regression target "
        "column was detected."
    )


    print(
        "\nNumeric columns that could be targets:"
    )


    numeric_candidates = []


    for col in df.columns:

        if pd.api.types.is_numeric_dtype(
            df[col]
        ):

            unique_count = df[col].nunique(
                dropna=True
            )


            if unique_count >= 2:

                numeric_candidates.append(
                    (
                        col,
                        unique_count
                    )
                )


    if numeric_candidates:

        for col, count in numeric_candidates:

            print(
                f"  {col} "
                f"-> {count} unique values"
            )

    else:

        raise ValueError(
            "No numeric column suitable "
            "for regression was found."
        )


    # --------------------------------------------------------
    # Display all columns
    # --------------------------------------------------------

    print(
        "\nAll dataset columns:"
    )


    for i, col in enumerate(df.columns):

        print(
            f"{i}: {col} "
            f"-> "
            f"{df[col].nunique(dropna=True)} "
            f"unique values "
            f"-> dtype={df[col].dtype}"
        )


    # --------------------------------------------------------
    # Ask user for target
    # --------------------------------------------------------

    target_input = input(
        "\nEnter TARGET column name "
        "or column number: "
    ).strip()


    # User entered column number

    if target_input.isdigit():

        index = int(target_input)


        if (
            index < 0
            or index >= len(df.columns)
        ):

            raise ValueError(
                "Invalid column number."
            )


        target_col = df.columns[index]


    # User entered column name

    else:

        matching_columns = [

            col

            for col in df.columns

            if str(col).strip()
            == target_input

        ]


        if not matching_columns:

            raise ValueError(
                f"Column '{target_input}' "
                "was not found."
            )


        target_col = matching_columns[0]


print("\n" + "=" * 70)
print("TARGET INFORMATION")
print("=" * 70)

print(
    "\nTarget column:",
    target_col
)


# ============================================================
# 6. TARGET MUST BE NUMERIC
# ============================================================

if not pd.api.types.is_numeric_dtype(
    df[target_col]
):

    raise ValueError(

        f"\nThe selected target '{target_col}' "
        "is not numeric.\n\n"

        "This experiment is for REGRESSION.\n"

        "Please choose a numeric target column."
    )


# ============================================================
# 7. REMOVE MISSING TARGET VALUES
# ============================================================

missing_target = df[target_col].isna().sum()


if missing_target > 0:

    print(
        f"\nRemoving {missing_target} "
        "rows with missing target."
    )

    df = df.dropna(
        subset=[target_col]
    )


# ============================================================
# 8. TARGET INFORMATION
# ============================================================

print(
    "\nTarget statistics:"
)

print(
    df[target_col].describe()
)


# ============================================================
# 9. SEPARATE FEATURES AND TARGET
# ============================================================

X_df = df.drop(
    columns=[target_col]
)

y = df[target_col]


# ============================================================
# 10. REMOVE ID-LIKE COLUMNS
# ============================================================

id_columns = []


for col in X_df.columns:

    name = (
        str(col)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )


    unique_count = X_df[col].nunique(
        dropna=True
    )


    # Common ID names

    if (

        name == "id"

        or name.endswith("_id")

        or name.startswith("id_")

        or name in [

            "studentid",
            "student_id",

            "customerid",
            "customer_id",

            "userid",
            "user_id",

            "transactionid",
            "transaction_id",

            "recordid",
            "record_id"
        ]

    ):

        id_columns.append(col)


    # Very high-cardinality text
    # often represents names / IDs

    elif (

        X_df[col].dtype == "object"

        and len(X_df) > 20

        and unique_count
        > 0.95 * len(X_df)

    ):

        id_columns.append(col)


if id_columns:

    print(
        "\nRemoving ID-like columns:"
    )


    for col in id_columns:

        print(
            " -",
            col
        )


    X_df = X_df.drop(
        columns=id_columns
    )


# ============================================================
# 11. REMOVE CONSTANT FEATURES
# ============================================================

constant_columns = []


for col in X_df.columns:

    if X_df[col].nunique(
        dropna=True
    ) <= 1:

        constant_columns.append(col)


if constant_columns:

    print(
        "\nRemoving constant columns:"
    )


    for col in constant_columns:

        print(
            " -",
            col
        )


    X_df = X_df.drop(
        columns=constant_columns
    )


# ============================================================
# 12. CHECK FEATURES
# ============================================================

if X_df.shape[1] == 0:

    raise ValueError(
        "No usable feature columns remain."
    )


# ============================================================
# 13. DETECT NUMERIC FEATURES
# ============================================================

numeric_features = (

    X_df

    .select_dtypes(
        include=[
            "int64",
            "int32",
            "int16",
            "int8",

            "float64",
            "float32",
            "float16"
        ]
    )

    .columns

    .tolist()
)


# ============================================================
# 14. DETECT CATEGORICAL FEATURES
# ============================================================

categorical_features = (

    X_df

    .select_dtypes(
        include=[
            "object",
            "category",
            "bool"
        ]
    )

    .columns

    .tolist()
)


print("\n" + "=" * 70)
print("FEATURE INFORMATION")
print("=" * 70)


print(
    "\nTotal features:",
    X_df.shape[1]
)


print(
    "\nNumeric features:",
    len(numeric_features)
)

print(
    numeric_features
)


print(
    "\nCategorical features:",
    len(categorical_features)
)

print(
    categorical_features
)


# ============================================================
# 15. TRAIN / TEST SPLIT
# ============================================================

Xtr, Xte, ytr, yte = train_test_split(

    X_df,

    y,

    test_size=0.25,

    random_state=42
)


print("\n" + "=" * 70)
print("TRAIN / TEST SPLIT")
print("=" * 70)


print(
    "\nTraining samples:",
    len(Xtr)
)


print(
    "Testing samples:",
    len(Xte)
)


# ============================================================
# 16. NUMERIC PREPROCESSOR
# ============================================================

numeric_pipeline = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    ),

    (
        "scaler",
        StandardScaler()
    )
])


# ============================================================
# 17. CATEGORICAL PREPROCESSOR
# ============================================================

categorical_pipeline = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),

    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            drop="first",
            sparse_output=False
        )
    )
])


# ============================================================
# 18. FULL PREPROCESSOR
# ============================================================

transformers = []


if numeric_features:

    transformers.append(

        (
            "num",

            numeric_pipeline,

            numeric_features
        )

    )


if categorical_features:

    transformers.append(

        (
            "cat",

            categorical_pipeline,

            categorical_features
        )

    )


if not transformers:

    raise ValueError(
        "No numeric or categorical features found."
    )


prep = ColumnTransformer(
    transformers=transformers
)


# ============================================================
# A. NUMERIC FEATURES ONLY
# ============================================================

print("\n")
print("=" * 70)
print("A. NUMERIC FEATURES ONLY")
print("=" * 70)


if len(numeric_features) == 0:

    print(
        "\nNo numeric features are available."
    )

    print(
        "This section cannot be performed."
    )

    r2_numeric = None

else:

    # --------------------------------------------------------
    # Numeric-only preprocessing
    # --------------------------------------------------------

    numeric_only_pipeline = Pipeline([

        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )

    ])


    Xtr_num = numeric_only_pipeline.fit_transform(
        Xtr[numeric_features]
    )


    Xte_num = numeric_only_pipeline.transform(
        Xte[numeric_features]
    )


    # --------------------------------------------------------
    # Linear regression
    # --------------------------------------------------------

    m1 = LinearRegression()


    m1.fit(
        Xtr_num,
        ytr
    )


    pred1 = m1.predict(
        Xte_num
    )


    r2_numeric = r2_score(
        yte,
        pred1
    )


    rmse_numeric = np.sqrt(
        mean_squared_error(
            yte,
            pred1
        )
    )


    mae_numeric = mean_absolute_error(
        yte,
        pred1
    )


    print(
        "\nNumeric-only R2 =",
        round(
            r2_numeric,
            4
        )
    )


    print(
        "Numeric-only RMSE =",
        round(
            rmse_numeric,
            4
        )
    )


    print(
        "Numeric-only MAE =",
        round(
            mae_numeric,
            4
        )
    )


# ============================================================
# B. NUMERIC + CATEGORICAL
# ============================================================

print("\n")
print("=" * 70)
print("B. NUMERIC + ONE-HOT ENCODED CATEGORICAL FEATURES")
print("=" * 70)


m2 = Pipeline([

    (
        "prep",
        prep
    ),

    (
        "lr",
        LinearRegression()
    )

])


m2.fit(
    Xtr,
    ytr
)


pred2 = m2.predict(
    Xte
)


r2_encoded = r2_score(
    yte,
    pred2
)


rmse_encoded = np.sqrt(
    mean_squared_error(
        yte,
        pred2
    )
)


mae_encoded = mean_absolute_error(
    yte,
    pred2
)


print(
    "\nNumeric + encoded R2 =",
    round(
        r2_encoded,
        4
    )
)


print(
    "Numeric + encoded RMSE =",
    round(
        rmse_encoded,
        4
    )
)


print(
    "Numeric + encoded MAE =",
    round(
        mae_encoded,
        4
    )
)


# ============================================================
# SHOW CREATED FEATURES
# ============================================================

print(
    "\nColumns created by the transformer:"
)


try:

    transformed_columns = (
        m2.named_steps["prep"]
        .get_feature_names_out()
    )


    print(
        list(transformed_columns)
    )


except Exception:

    print(
        "Could not retrieve transformed "
        "column names."
    )


# ============================================================
# C. WHY SCALING MATTERS
# COEFFICIENT COMPARISON
# ============================================================

print("\n")
print("=" * 70)
print("C. SCALING AND COEFFICIENT COMPARISON")
print("=" * 70)


if len(numeric_features) == 0:

    print(
        "\nNo numeric features available."
    )

else:

    # --------------------------------------------------------
    # Impute numeric values
    # --------------------------------------------------------

    numeric_imputer = SimpleImputer(
        strategy="median"
    )


    Xtr_num_clean = (
        numeric_imputer
        .fit_transform(
            Xtr[numeric_features]
        )
    )


    # --------------------------------------------------------
    # Unscaled model
    # --------------------------------------------------------

    raw = LinearRegression()


    raw.fit(
        Xtr_num_clean,
        ytr
    )


    # --------------------------------------------------------
    # Scaled model
    # --------------------------------------------------------

    scaled_model = Pipeline([

        (
            "sc",
            StandardScaler()
        ),

        (
            "lr",
            LinearRegression()
        )

    ])


    scaled_model.fit(
        Xtr_num_clean,
        ytr
    )


    # --------------------------------------------------------
    # Raw coefficients
    # --------------------------------------------------------

    print(
        "\nUnscaled coefficients "
        "(original units):"
    )


    raw_coefficients = pd.Series(

        raw.coef_,

        index=numeric_features
    )


    print(
        raw_coefficients
        .round(4)
        .to_string()
    )


    # --------------------------------------------------------
    # Scaled coefficients
    # --------------------------------------------------------

    print(
        "\nScaled coefficients "
        "(per 1 standard deviation):"
    )


    scaled_coefficients = pd.Series(

        scaled_model[
            "lr"
        ].coef_,

        index=numeric_features
    )


    print(
        scaled_coefficients
        .round(4)
        .to_string()
    )


    # --------------------------------------------------------
    # Compare absolute coefficient sizes
    # --------------------------------------------------------

    comparison = pd.DataFrame({

        "unscaled": raw_coefficients,

        "scaled": scaled_coefficients,

        "abs_scaled": (
            scaled_coefficients
            .abs()
        )

    })


    comparison = comparison.sort_values(
        "abs_scaled",
        ascending=False
    )


    print(
        "\nCoefficient comparison:"
    )


    print(
        comparison
        .round(4)
        .to_string()
    )


# ============================================================
# D. SCALING AND SGD CONVERGENCE
# ============================================================

print("\n")
print("=" * 70)
print("D. SCALING AND SGD CONVERGENCE")
print("=" * 70)


if len(numeric_features) == 0:

    print(
        "\nNo numeric features available."
    )

else:

    # --------------------------------------------------------
    # Prepare numeric data
    # --------------------------------------------------------

    sgd_imputer = SimpleImputer(
        strategy="median"
    )


    Xtr_sgd = (
        sgd_imputer
        .fit_transform(
            Xtr[numeric_features]
        )
    )


    # --------------------------------------------------------
    # Unscaled data
    # --------------------------------------------------------

    Xtr_unscaled = Xtr_sgd


    # --------------------------------------------------------
    # Scaled data
    # --------------------------------------------------------

    sgd_scaler = StandardScaler()


    Xtr_scaled = (
        sgd_scaler
        .fit_transform(
            Xtr_sgd
        )
    )


    # --------------------------------------------------------
    # Train SGD on unscaled data
    # --------------------------------------------------------

    t0 = time.time()


    sgd_unscaled = SGDRegressor(

        max_iter=5000,

        tol=1e-4,

        random_state=0
    )


    sgd_unscaled.fit(
        Xtr_unscaled,
        ytr
    )


    unscaled_time = (
        time.time()
        -
        t0
    )


    # --------------------------------------------------------
    # Train SGD on scaled data
    # --------------------------------------------------------

    t0 = time.time()


    sgd_scaled = SGDRegressor(

        max_iter=5000,

        tol=1e-4,

        random_state=0
    )


    sgd_scaled.fit(
        Xtr_scaled,
        ytr
    )


    scaled_time = (
        time.time()
        -
        t0
    )


    # --------------------------------------------------------
    # Results
    # --------------------------------------------------------

    print(
        f"\nSGD on unscaled "
        f"iterations = "
        f"{sgd_unscaled.n_iter_}"
    )


    print(
        f"SGD on unscaled "
        f"time = "
        f"{unscaled_time:.3f}s"
    )


    print(
        f"\nSGD on scaled "
        f"iterations = "
        f"{sgd_scaled.n_iter_}"
    )


    print(
        f"SGD on scaled "
        f"time = "
        f"{scaled_time:.3f}s"
    )


# ============================================================
# E. FINAL COMPARISON
# ============================================================

print("\n")
print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)


print(
    f"\nDataset:"
)

print(
    file_path
)


print(
    f"\nTarget:"
)

print(
    target_col
)


print(
    f"\nRows:"
)

print(
    len(df)
)


print(
    f"\nFeatures:"
)

print(
    X_df.shape[1]
)


print(
    "\nNumeric features:"
)

print(
    len(numeric_features)
)


print(
    "Categorical features:"
)

print(
    len(categorical_features)
)


print(
    "\n-------------------------------"
)

if r2_numeric is not None:

    print(
        "Numeric-only R2:",
        round(
            r2_numeric,
            4
        )
    )

else:

    print(
        "Numeric-only R2: N/A"
    )


print(
    "Numeric + encoded R2:",
    round(
        r2_encoded,
        4
    )
)


print(
    "Numeric + encoded RMSE:",
    round(
        rmse_encoded,
        4
    )
)


print(
    "Numeric + encoded MAE:",
    round(
        mae_encoded,
        4
    )
)


print(
    "\n-------------------------------"
)


if len(numeric_features) > 0:

    print(
        "SGD unscaled iterations:",
        sgd_unscaled.n_iter_
    )


    print(
        "SGD scaled iterations:",
        sgd_scaled.n_iter_
    )


    print(
        "SGD unscaled time:",
        round(
            unscaled_time,
            3
        ),
        "seconds"
    )


    print(
        "SGD scaled time:",
        round(
            scaled_time,
            3
        ),
        "seconds"
    )


print("\n")
print("=" * 70)
print("EXPERIMENT COMPLETED SUCCESSFULLY!")
print("=" * 70)

UNIVERSAL REGRESSION EXPERIMENT

Upload your CSV dataset.


Saving placement_predict_50k Dataset.csv to placement_predict_50k Dataset.csv

Selected file:
placement_predict_50k Dataset.csv

Dataset loaded successfully!

Dataset shape:
(50000, 32)

First 5 rows:
   StudentID  Gender       City CollegeTier Stream Specialisation Hostel  \
0          1    Male  Ahmedabad       Tier2    ECE     Networking     No   
1          2  Female     Mumbai       Tier2    ECE    DataScience    Yes   
2          3    Male    Kolkata       Tier2     IT    DataScience    Yes   
3          4    Male     Jaipur       Tier1     CS             AI     No   
4          5    Male       Pune       Tier2     IT    DataScience    Yes   

  HistoryOfBacklogs  SGPA_Sem1  SGPA_Sem2  ...  Publications  \
0                No       6.02       6.54  ...             0   
1               Yes       5.84       5.12  ...             0   
2                No       4.91       5.29  ...             0   
3                No       7.67       8.03  ...             0   
4                No   